# 🎞 AI motion server on a free GPU (Kaggle / Colab)
Makes your scene pictures move for real (image-to-video) for the **Video Generation** extension.

1. **Kaggle**: Settings → Accelerator → **GPU T4 ×2**, and **Internet: On**. **Colab**: Runtime → Change runtime type → **T4 GPU**.
2. Run every cell from top to bottom.
3. The last cell prints a **link** and a **token**: paste both in the extension (Video tab → 🎞 AI motion).
4. Keep this tab open while the clips are made.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: turn the GPU on in the settings first'

In [ ]:
# Settings
MODEL = 'ltx'        # 'ltx' (fast, fits a T4) or 'wan' (better, needs 24 GB+)
LIPSYNC = False      # True = also install LatentSync lip-sync (+10 min install, needs front-facing faces)
PORT = 7871

In [ ]:
import os
os.makedirs('motion/backends', exist_ok=True)
%cd motion

In [ ]:
%%writefile server.py
"""
AI motion server for the "Video Generation" Chrome extension (v4.3).
Turns each scene picture into a real moving video clip (image-to-video) on a GPU,
optionally with lip-sync to the scene's voice. Runs on your own PC or on a free
Kaggle / Colab GPU (see kaggle_colab.ipynb).

  python server.py --model ltx  --port 7871               (fast, ~12 GB VRAM)
  python server.py --model wan  --port 7871               (better, ~24 GB VRAM)
  python server.py --model mock --port 7871               (no GPU, only to test the connection)
  add --lipsync to sync the speaker's lips to the voice (LatentSync, install_lipsync first)
  add --token SECRET to require that token (always do this on Kaggle / Colab)

Endpoints (JSON):
  GET  /health            -> {"ok", "model", "ready", "device", "vram_gb", "lipsync", "error", "queue"}
  POST /animate           -> body {"image": b64, "prompt", "negative", "seconds", "width", "height",
                              "audio": b64 wav (optional), "lipsync": bool, "seed"}  -> {"job": id}
  GET  /job?id=ID         -> {"state": queued|running|done|error, "progress": 0-1, "message", "error"}
  GET  /result?id=ID      -> video/mp4
Only the standard library here; each model's libraries are imported by its backend.
"""
import argparse
import base64
import importlib
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import traceback
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
WORK = os.path.join(HERE, "motion_jobs")

STATE = {"model": None, "backend": None, "ready": False, "error": None, "device": "cpu", "vram_gb": 0,
         "lipsync": False, "lipsync_error": None, "token": ""}
JOBS = {}
QUEUE = []
QLOCK = threading.Condition()


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def ffmpeg_bin():
    for c in (os.environ.get("FFMPEG"), shutil.which("ffmpeg")):
        if c:
            return c
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return "ffmpeg"


def run(cmd):
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0:
        raise RuntimeError(" ".join(cmd[:3]) + " failed: " + p.stdout[-800:])
    return p.stdout


def last_frame(video, out_png):
    run([ffmpeg_bin(), "-loglevel", "error", "-y", "-sseof", "-0.1", "-i", video, "-frames:v", "1", out_png])


def concat(parts, out):
    lst = out + ".txt"
    with open(lst, "w") as f:
        for p in parts:
            f.write("file '%s'\n" % p.replace("\\", "/"))
    run([ffmpeg_bin(), "-loglevel", "error", "-y", "-f", "concat", "-safe", "0", "-i", lst,
         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", out])


def trim(src, seconds, out):
    run([ffmpeg_bin(), "-loglevel", "error", "-y", "-i", src, "-t", "%.3f" % seconds, "-an",
         "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18", out])


# ---------------------------------------------------------------- the job worker
def do_job(job):
    d = job["dir"]
    req = job["req"]
    img = os.path.join(d, "in.png")
    with open(img, "wb") as f:
        f.write(base64.b64decode(req["image"]))
    audio = None
    if req.get("audio"):
        audio = os.path.join(d, "voice.wav")
        with open(audio, "wb") as f:
            f.write(base64.b64decode(req["audio"]))
    seconds = max(1.0, min(60.0, float(req.get("seconds") or 5)))
    B = STATE["backend"]
    seg_len = B.SEGMENT_SECONDS
    n_seg = max(1, int(-(-seconds // seg_len)))  # ceil
    parts, start = [], img
    for k in range(n_seg):
        job["message"] = "clip %d/%d" % (k + 1, n_seg)

        def prog(p, k=k):
            job["progress"] = min(0.95, (k + p) / n_seg * (0.8 if audio and req.get("lipsync") else 1.0))

        out = os.path.join(d, "seg%02d.mp4" % k)
        B.animate(start, req.get("prompt") or "", req.get("negative") or "", out,
                  width=int(req.get("width") or 0), height=int(req.get("height") or 0),
                  seed=int(req.get("seed") or 0) + k, progress=prog)
        parts.append(out)
        if k < n_seg - 1:  # agla hissa pichhle ke aakhri frame se — ek hi lambi chaal
            start = os.path.join(d, "last%02d.png" % k)
            last_frame(out, start)
    joined = os.path.join(d, "joined.mp4")
    concat(parts, joined) if len(parts) > 1 else shutil.copy(parts[0], joined)
    final = os.path.join(d, "final.mp4")
    trim(joined, seconds, final)
    if audio and req.get("lipsync"):
        if STATE["lipsync"]:
            job["message"] = "lip-sync"
            try:
                import lipsync
                synced = os.path.join(d, "synced.mp4")
                lipsync.sync(final, audio, synced)
                trim(synced, seconds, final)
                job["lipsynced"] = True
            except Exception as e:
                job["warning"] = "lip-sync skipped: %s" % str(e)[:300]
                log("lip-sync failed:", e)
        else:
            job["warning"] = "lip-sync is not installed on this server"
    job["result"] = final
    job["progress"] = 1.0


def worker():
    while True:
        with QLOCK:
            while not QUEUE:
                QLOCK.wait()
            jid = QUEUE.pop(0)
        job = JOBS.get(jid)
        if not job:
            continue
        job["state"] = "running"
        t0 = time.time()
        try:
            do_job(job)
            job["state"] = "done"
            job["message"] = "done in %ds" % (time.time() - t0)
            log("job", jid, "done in %.0fs" % (time.time() - t0))
        except Exception as e:
            traceback.print_exc()
            job["state"] = "error"
            job["error"] = str(e)[:500]


def cleanup_old():
    now = time.time()
    for jid, j in list(JOBS.items()):
        if now - j["created"] > 6 * 3600:
            shutil.rmtree(j["dir"], ignore_errors=True)
            JOBS.pop(jid, None)


# ---------------------------------------------------------------- HTTP
class H(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _cors(self):
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "content-type, x-token")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")

    def _json(self, code, obj):
        b = json.dumps(obj).encode()
        self.send_response(code)
        self._cors()
        self.send_header("content-type", "application/json")
        self.send_header("content-length", str(len(b)))
        self.end_headers()
        self.wfile.write(b)

    def _auth(self):
        tok = STATE["token"]
        if tok and self.headers.get("x-token", "") != tok:
            self._json(401, {"ok": False, "error": "wrong or missing token"})
            return False
        return True

    def do_OPTIONS(self):
        self.send_response(204)
        self._cors()
        self.end_headers()

    def do_GET(self):
        path, _, q = self.path.partition("?")
        args = dict(p.split("=", 1) for p in q.split("&") if "=" in p)
        if path == "/health":
            if not self._auth():
                return
            return self._json(200, {"ok": True, "model": STATE["model"], "ready": STATE["ready"], "device": STATE["device"],
                                    "vram_gb": STATE["vram_gb"], "lipsync": STATE["lipsync"], "lipsync_error": STATE["lipsync_error"],
                                    "error": STATE["error"], "queue": len(QUEUE),
                                    "segment_seconds": getattr(STATE["backend"], "SEGMENT_SECONDS", None)})
        if not self._auth():
            return
        job = JOBS.get(args.get("id", ""))
        if path == "/job":
            if not job:
                return self._json(404, {"state": "error", "error": "unknown job (server restarted?)"})
            pos = QUEUE.index(job["id"]) + 1 if job["id"] in QUEUE else 0
            return self._json(200, {"state": job["state"], "progress": job["progress"], "message": job.get("message", ""),
                                    "error": job.get("error"), "warning": job.get("warning"), "lipsynced": job.get("lipsynced", False),
                                    "queue_position": pos})
        if path == "/result":
            if not job or job["state"] != "done":
                return self._json(404, {"error": "not ready"})
            with open(job["result"], "rb") as f:
                b = f.read()
            self.send_response(200)
            self._cors()
            self.send_header("content-type", "video/mp4")
            self.send_header("content-length", str(len(b)))
            self.end_headers()
            self.wfile.write(b)
            return
        self._json(404, {"error": "not found"})

    def do_POST(self):
        if not self._auth():
            return
        if self.path.split("?")[0] != "/animate":
            return self._json(404, {"error": "not found"})
        if not STATE["ready"]:
            return self._json(503, {"error": "model not ready: %s" % (STATE["error"] or "still loading")})
        n = int(self.headers.get("content-length", "0"))
        try:
            req = json.loads(self.rfile.read(n) or b"{}")
        except Exception:
            return self._json(400, {"error": "bad JSON"})
        if not req.get("image"):
            return self._json(400, {"error": "image missing"})
        cleanup_old()
        jid = uuid.uuid4().hex[:12]
        d = os.path.join(WORK, jid)
        os.makedirs(d, exist_ok=True)
        JOBS[jid] = {"id": jid, "dir": d, "req": req, "state": "queued", "progress": 0.0, "created": time.time()}
        with QLOCK:
            QUEUE.append(jid)
            QLOCK.notify()
        log("job", jid, "queued · %.1fs · %s" % (float(req.get("seconds") or 5), (req.get("prompt") or "")[:70]))
        self._json(200, {"job": jid})


def load(model, device, want_lipsync):
    try:
        if device != "cpu":
            try:
                import torch
                if torch.cuda.is_available():
                    STATE["device"] = "cuda"
                    STATE["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
            except Exception:
                pass
        mod = importlib.import_module("backends." + model)
        log("loading %s on %s (first time downloads the model, this can take a while)..." % (model, STATE["device"]))
        mod.load(STATE["device"])
        STATE["backend"] = mod
        STATE["ready"] = True
        log("%s ready" % model)
    except Exception as e:
        traceback.print_exc()
        STATE["error"] = "%s: %s" % (type(e).__name__, str(e)[:400])
    if want_lipsync:
        try:
            import lipsync
            lipsync.check()
            STATE["lipsync"] = True
            log("lip-sync (LatentSync) ready")
        except Exception as e:
            STATE["lipsync_error"] = str(e)[:300]
            log("lip-sync not available:", e)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="ltx", choices=["ltx", "wan", "mock"])
    ap.add_argument("--port", type=int, default=7871)
    ap.add_argument("--host", default="127.0.0.1")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--lipsync", action="store_true")
    ap.add_argument("--token", default=os.environ.get("MOTION_TOKEN", ""))
    a = ap.parse_args()
    STATE["model"] = a.model
    STATE["token"] = a.token
    os.makedirs(WORK, exist_ok=True)
    threading.Thread(target=load, args=(a.model, a.device, a.lipsync), daemon=True).start()
    threading.Thread(target=worker, daemon=True).start()
    log("AI motion server on http://%s:%d  (model: %s%s)" % (a.host, a.port, a.model, ", token required" if a.token else ""))
    ThreadingHTTPServer((a.host, a.port), H).serve_forever()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile lipsync.py
"""Lip-sync with LatentSync (ByteDance, Apache-2.0): makes the speaker's lips follow the scene's voice.
Works best on realistic / 3D-style faces that face the camera; flat comic faces may not be found (then the clip
is used without lip-sync). Installed by install_lipsync.bat (or the Kaggle / Colab notebook) into ./LatentSync"""
import os
import subprocess
import sys

DIR = os.environ.get("LATENTSYNC_DIR", os.path.join(os.path.dirname(os.path.abspath(__file__)), "LatentSync"))
CONFIG = os.environ.get("LATENTSYNC_CONFIG", "configs/unet/stage2.yaml")
CKPT = os.environ.get("LATENTSYNC_CKPT", "checkpoints/latentsync_unet.pt")
PY = os.environ.get("LATENTSYNC_PY", sys.executable)


def check():
    for p in (DIR, os.path.join(DIR, CKPT), os.path.join(DIR, "checkpoints", "whisper", "tiny.pt")):
        if not os.path.exists(p):
            raise RuntimeError("LatentSync not installed (missing %s)" % p)


def sync(video, audio, out):
    cmd = [PY, "-m", "scripts.inference", "--unet_config_path", CONFIG, "--inference_ckpt_path", CKPT,
           "--inference_steps", os.environ.get("LATENTSYNC_STEPS", "20"), "--guidance_scale", "1.5",
           "--video_path", os.path.abspath(video), "--audio_path", os.path.abspath(audio), "--video_out_path", os.path.abspath(out)]
    p = subprocess.run(cmd, cwd=DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode != 0 or not os.path.exists(out):
        tail = p.stdout[-600:]
        if "face" in tail.lower():
            raise RuntimeError("no face found for lip-sync")
        raise RuntimeError(tail)


In [ ]:
%%writefile backends/__init__.py


In [ ]:
%%writefile backends/common.py
"""Shared helpers for the image-to-video backends."""
from PIL import Image


def fit_image(path, long_side, multiple, width=0, height=0):
    """Open the scene picture and centre-crop / resize it to a size the model accepts."""
    im = Image.open(path).convert("RGB")
    if width and height:
        W, H = width, height
    else:
        ar = im.width / im.height
        if ar >= 1:
            W, H = long_side, long_side / ar
        else:
            W, H = long_side * ar, long_side
    W = max(multiple, int(round(W / multiple)) * multiple)
    H = max(multiple, int(round(H / multiple)) * multiple)
    # centre-crop to the target aspect, then resize
    tar = W / H
    if im.width / im.height > tar:
        nw = int(im.height * tar)
        im = im.crop(((im.width - nw) // 2, 0, (im.width - nw) // 2 + nw, im.height))
    else:
        nh = int(im.width / tar)
        im = im.crop((0, (im.height - nh) // 2, im.width, (im.height - nh) // 2 + nh))
    return im.resize((W, H), Image.LANCZOS), W, H


def step_callback(total, progress):
    def cb(pipe, step, timestep, kwargs):
        try:
            progress((step + 1) / max(1, total))
        except Exception:
            pass
        return kwargs
    return cb


NEGATIVE = ("worst quality, low quality, blurry, distorted face, deformed hands, extra fingers, extra limbs, morphing, "
            "flickering, jittery, text, watermark, sudden change of style, new characters appearing")


In [ ]:
%%writefile backends/mock.py
"""No GPU needed: makes a slow zoom clip with ffmpeg. Only to test the extension <-> server connection."""
import subprocess
import time

from server import ffmpeg_bin

SEGMENT_SECONDS = 5.0


def load(device):
    pass


def animate(image, prompt, negative, out, width=0, height=0, seed=0, progress=lambda p: None):
    for k in range(5):
        progress(k / 5)
        time.sleep(0.2)
    fps, n = 24, int(SEGMENT_SECONDS * 24)
    vf = ("scale=1280:720:force_original_aspect_ratio=increase,crop=1280:720,"
          "zoompan=z='1+0.0015*on':x='iw/2-(iw/zoom/2)+sin(on/9)*6':y='ih/2-(ih/zoom/2)':d=%d:s=1280x720:fps=%d" % (n, fps))
    subprocess.run([ffmpeg_bin(), "-loglevel", "error", "-y", "-loop", "1", "-i", image, "-vf", vf, "-frames:v", str(n),
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", out], check=True)
    progress(1)


In [ ]:
%%writefile backends/ltx.py
"""LTX-Video image-to-video (Lightricks, open weights). Fast: ~5 s clip in a few minutes on a 12-16 GB GPU.
Model: https://huggingface.co/Lightricks/LTX-Video  (check its licence before monetising)."""
import os

import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

from backends.common import NEGATIVE, fit_image, step_callback

MODEL_ID = os.environ.get("LTX_MODEL", "Lightricks/LTX-Video")
SEGMENT_SECONDS = 5.0
FPS = 24
STEPS = int(os.environ.get("LTX_STEPS", "40"))
PIPE = None


def load(device):
    global PIPE
    bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
    dtype = torch.bfloat16 if bf16 else (torch.float16 if device == "cuda" else torch.float32)
    PIPE = LTXImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
    if device == "cuda":
        vram = torch.cuda.get_device_properties(0).total_memory / 2**30
        if vram < 20:
            PIPE.enable_model_cpu_offload()  # 12-16 GB cards (Kaggle / Colab T4)
        else:
            PIPE.to("cuda")
        PIPE.vae.enable_tiling()
    else:
        PIPE.to("cpu")


def animate(image, prompt, negative, out, width=0, height=0, seed=0, progress=lambda p: None):
    im, W, H = fit_image(image, int(os.environ.get("LTX_SIZE", "768")), 32, width, height)
    frames = int(SEGMENT_SECONDS * FPS) // 8 * 8 + 1  # LTX: 8k + 1 frames
    g = torch.Generator(device="cpu").manual_seed(seed or 42)
    video = PIPE(image=im, prompt=prompt, negative_prompt=negative or NEGATIVE, width=W, height=H, num_frames=frames,
                 num_inference_steps=STEPS, generator=g, callback_on_step_end=step_callback(STEPS, progress)).frames[0]
    export_to_video(video, out, fps=FPS)


In [ ]:
%%writefile backends/wan.py
"""Wan 2.1 image-to-video 480p (Alibaba, Apache-2.0). Better motion and faces than LTX, but slow and needs a big GPU
(24 GB+; works on 16 GB only very slowly with offloading). Model: https://huggingface.co/Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"""
import os

import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video
from transformers import CLIPVisionModel

from backends.common import NEGATIVE, fit_image, step_callback

MODEL_ID = os.environ.get("WAN_MODEL", "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers")
SEGMENT_SECONDS = 5.0
FPS = 16
STEPS = int(os.environ.get("WAN_STEPS", "30"))
PIPE = None


def load(device):
    global PIPE
    dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
    image_encoder = CLIPVisionModel.from_pretrained(MODEL_ID, subfolder="image_encoder", torch_dtype=torch.float32)
    vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    PIPE = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, image_encoder=image_encoder, torch_dtype=dtype)
    if device == "cuda":
        PIPE.enable_model_cpu_offload()


def animate(image, prompt, negative, out, width=0, height=0, seed=0, progress=lambda p: None):
    im, W, H = fit_image(image, 832, 16, width, height)
    frames = int(SEGMENT_SECONDS * FPS) // 4 * 4 + 1  # Wan: 4k + 1 frames
    g = torch.Generator(device="cpu").manual_seed(seed or 42)
    video = PIPE(image=im, prompt=prompt, negative_prompt=negative or NEGATIVE, height=H, width=W, num_frames=frames,
                 guidance_scale=5.0, num_inference_steps=STEPS, generator=g, callback_on_step_end=step_callback(STEPS, progress)).frames[0]
    export_to_video(video, out, fps=FPS)


In [ ]:
!pip -q install 'diffusers>=0.33' transformers accelerate sentencepiece ftfy imageio imageio-ffmpeg
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared

In [ ]:
# Optional lip-sync (LatentSync) in its own environment so its versions do not clash
if LIPSYNC:
    !git clone -q https://github.com/bytedance/LatentSync.git
    !python -m venv LatentSync/venv && LatentSync/venv/bin/pip -q install -r LatentSync/requirements.txt huggingface_hub
    !cd LatentSync && venv/bin/huggingface-cli download ByteDance/LatentSync-1.5 --local-dir checkpoints --exclude "*.git*" "README.md"
    os.environ['LATENTSYNC_PY'] = os.path.abspath('LatentSync/venv/bin/python')

In [ ]:
# Start the server + a public link
import subprocess, secrets, time, re, os
TOKEN = secrets.token_urlsafe(12)
args = ['python', 'server.py', '--model', MODEL, '--port', str(PORT), '--token', TOKEN] + (['--lipsync'] if LIPSYNC else [])
srv = subprocess.Popen(args, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('=' * 60)
print('Server link :', url)
print('Token       :', TOKEN)
print('=' * 60)
print('Paste both in the extension: Video tab -> 🎞 AI motion. The model loads now (first time: download, several minutes).')

In [ ]:
# Watch the server (run again any time to see the latest lines)
!tail -n 25 server.log

Keep this tab open while the extension makes the clips. If the session stops, run the cells again: you get a new link and token.